# Support Response Fine-Tuning

**Goal:** build a practical fine-tuning pipeline for improving the **style** of customer support responses, not the factual policy layer.

This notebook shows how to:

- structure support conversations into supervised fine-tuning examples
- create a train/validation split that avoids misleading leakage
- define offline and online evaluation for style quality
- decide when fine-tuning is worth doing versus prompt engineering or RAG

## Problem Framing

Support teams often want a response style that is:

- empathetic but concise
- clear about next steps
- consistent across agents and channels
- on-brand without sounding robotic

That is a good fine-tuning target because **style is stable** even when factual knowledge changes.

## What This Notebook Fine-Tunes

The model learns a support-response style from examples shaped like:

`customer message + case facts + style policy -> target support reply`

## What This Notebook Does Not Fine-Tune

It does **not** attempt to teach volatile business facts such as refund windows, pricing, or outage status. Those should stay in retrieval or application logic.

---

## High-Level Pipeline

1. Define a support style guide
2. Build a supervised dataset of ideal replies
3. Format examples into chat-style SFT records
4. Split train/validation carefully
5. Fine-tune with LoRA on a local GPU
6. Evaluate style adherence, safety, and customer usefulness
7. Decide whether the gain justifies the maintenance cost

## 1. When Fine-Tuning Is the Right Tool

Fine-tuning is not the default answer. Use it when the main problem is **consistent generation behavior**, not missing knowledge.

| Need | Better Tool | Why |
|---|---|---|
| Model should sound more empathetic, concise, or on-brand every time | Fine-tuning | This is stable behavioral preference |
| Model needs fresh product facts or changing policies | RAG / tools | Facts change too often for weights |
| Team wants a small number of reusable prompt rules | Prompt engineering | Cheaper and faster to iterate |
| Output must follow a rigid schema | Structured decoding / validators | More reliable than training style alone |
| Agents disagree on tone across thousands of tickets | Fine-tuning | Weight updates improve consistency at scale |

### Good Fine-Tuning Signals

- you already have hundreds or thousands of high-quality exemplar replies
- the target tone is stable across time and channels
- prompt-only behavior still drifts in production
- the same style rules are repeated in every prompt
- latency or token cost matters enough to reduce long instructions

### Bad Fine-Tuning Signals

- the real problem is weak retrieval or bad policies
- your "gold" replies are inconsistent across agents
- policies change weekly
- you cannot measure success beyond "it feels better"
- you only have a few dozen examples

In [ ]:
# Install the packages used in this notebook.
# bitsandbytes is optional on some Windows setups; if it fails, keep the rest and use a non-4bit path.
!pip install -q pandas numpy scikit-learn datasets transformers peft trl accelerate evaluate rouge-score

In [ ]:
import json
import random
import re
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from datasets import Dataset
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

PROJECT_DIR = Path.cwd()
ARTIFACT_DIR = PROJECT_DIR / "artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)

RUN_TRAINING = False
RUN_OFFLINE_GENERATION_EVAL = False
BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
OUTPUT_DIR = ARTIFACT_DIR / "support-style-lora"

print(f"Project dir: {PROJECT_DIR}")
print(f"Artifacts dir: {ARTIFACT_DIR}")
print(f"Base model: {BASE_MODEL}")
print(f"Training enabled: {RUN_TRAINING}")
print(f"Generation eval enabled: {RUN_OFFLINE_GENERATION_EVAL}")

## 2. Define the Support Style Policy

The model should learn **how** to reply, not **what the current policy is**. Keep the style policy compact and stable.

### Example Style Guide

- acknowledge the customer's issue in the first sentence
- use plain language; avoid internal jargon
- state the next action clearly
- do not blame the customer
- do not overpromise timelines
- end with a helpful, calm closing
- when facts are missing, explain what is needed instead of guessing

A useful mental split is:

- **weights learn style**
- **retrieval/tools provide facts**

In [ ]:
STYLE_GUIDE = {
    "tone": [
        "warm",
        "calm",
        "clear",
        "professional",
        "non-defensive",
    ],
    "must_do": [
        "acknowledge the issue early",
        "state the most relevant next step",
        "use concise sentences",
        "set expectations honestly",
        "close with a helpful offer",
    ],
    "must_not_do": [
        "blame the customer",
        "invent missing facts",
        "use sarcasm",
        "promise outcomes you cannot control",
        "copy dense policy text verbatim",
    ],
}

STYLE_SYSTEM_PROMPT = '''You are a customer support assistant writing high-quality support replies.
Follow this style policy:
- Be empathetic, calm, and concise.
- Acknowledge the customer's issue in the first sentence.
- Explain the next step clearly.
- Use plain language, not internal jargon.
- Never invent policy details that are not present in the provided facts.
- Never blame the customer.
- End with a short, helpful closing.'''

print(json.dumps(STYLE_GUIDE, indent=2))

## 3. Build a Supervised Dataset

For demonstration, this notebook creates a synthetic dataset of support cases. In a real project, your best source is:

- past replies that were manually QA-approved
- edited responses from your strongest agents
- escalation resolutions rewritten into ideal customer-facing language

### Dataset Schema

Each row should separate:

- the **customer message**
- the **case facts** available to the agent
- the **target response** you want the model to imitate
- tags for slice analysis such as issue type or channel

That separation matters. If facts and style are mixed together sloppily, you cannot later diagnose whether the model failed on retrieval, policy grounding, or tone.

In [ ]:
records = [
    {
        "ticket_id": "T-1001",
        "issue_type": "refund_delay",
        "channel": "email",
        "region": "us",
        "customer_message": "Hi, I canceled my order five days ago and still haven't seen the refund on my card. Can you tell me what's going on?",
        "facts": [
            "Order was canceled successfully.",
            "Refunds typically appear within 5 to 10 business days depending on the bank.",
            "No manual action is currently required from the customer.",
        ],
        "target_response": "Hi, I'm sorry the refund still hasn't appeared yet. I checked the case and the cancellation went through successfully. Refunds usually show up within 5 to 10 business days depending on your bank, so you're still within the normal window right now. If it still isn't visible after that period, reply here and we'll investigate it with you. Thanks for your patience.",
    },
    {
        "ticket_id": "T-1002",
        "issue_type": "password_reset",
        "channel": "chat",
        "region": "us",
        "customer_message": "Your password reset email never shows up for me. I've tried three times.",
        "facts": [
            "Password reset emails can take a few minutes to arrive.",
            "Customer should check spam or promotions folders.",
            "The account email on file ends with @acme-example.com.",
        ],
        "target_response": "I'm sorry the reset email hasn't reached you yet. It can take a few minutes to arrive, and it may land in spam or promotions, so please check those folders first. I also confirmed the account is tied to the email ending in @acme-example.com. If you still don't see it after a few minutes, let us know and we can help with the next step.",
    },
    {
        "ticket_id": "T-1003",
        "issue_type": "shipment_delay",
        "channel": "email",
        "region": "eu",
        "customer_message": "My package has been stuck on 'label created' since Monday. Is it lost?",
        "facts": [
            "Carrier has not scanned the parcel yet.",
            "Label created status does not confirm loss.",
            "If no carrier scan appears after 3 business days, support can open a carrier inquiry.",
        ],
        "target_response": "I understand why that status is frustrating. Right now, 'label created' means the carrier has not scanned the parcel yet, so it does not confirm that the package is lost. If there is still no carrier scan after 3 business days, we can open an inquiry for you. If you'd like, reply after that point and we'll take the next step with you.",
    },
    {
        "ticket_id": "T-1004",
        "issue_type": "billing_error",
        "channel": "chat",
        "region": "us",
        "customer_message": "I was charged twice for the same subscription renewal.",
        "facts": [
            "Two charges are visible on the account.",
            "One charge is already marked pending reversal.",
            "Pending reversals can take 3 to 5 business days to disappear.",
        ],
        "target_response": "I'm sorry about the duplicate charge. I checked the account and can see both charges, and one of them is already marked for reversal. Pending reversals usually disappear within 3 to 5 business days. If the extra charge is still there after that window, reply here and we'll review it again with you.",
    },
    {
        "ticket_id": "T-1005",
        "issue_type": "account_lock",
        "channel": "email",
        "region": "apac",
        "customer_message": "My account says it's locked after too many login attempts. I need access today.",
        "facts": [
            "Account lock lasts 30 minutes after repeated failed attempts.",
            "Support cannot manually bypass the security lock.",
            "Password reset is available once the lock window ends.",
        ],
        "target_response": "I'm sorry you're locked out right when you need access. The account is under a 30-minute security lock after repeated failed attempts, and we can't manually bypass that lock. Once that window ends, you can use the password reset flow if needed. I know that's inconvenient, and if you run into trouble after the lock clears, we're here to help.",
    },
    {
        "ticket_id": "T-1006",
        "issue_type": "feature_request",
        "channel": "chat",
        "region": "us",
        "customer_message": "Do you support exporting dashboards to PowerPoint yet?",
        "facts": [
            "PowerPoint export is not currently available.",
            "PDF export is available on Pro and Enterprise plans.",
            "Product team reviews feature requests monthly.",
        ],
        "target_response": "Thanks for asking about this. PowerPoint export isn't available right now. If you need a presentation-ready option today, PDF export is available on Pro and Enterprise plans. I've also noted your request so it can be included in the product team's monthly review. If you want, I can point you to the current export options.",
    },
    {
        "ticket_id": "T-1007",
        "issue_type": "refund_delay",
        "channel": "chat",
        "region": "eu",
        "customer_message": "I canceled during the trial but I still see a charge on my statement.",
        "facts": [
            "Trial cancellation was recorded before renewal.",
            "The visible bank line is a temporary authorization, not a captured charge.",
            "Authorizations typically fall off in a few business days.",
        ],
        "target_response": "I understand why that looks concerning. I checked the account and your trial was canceled before renewal. The bank line you're seeing is a temporary authorization rather than a captured charge, and those usually fall off within a few business days. If it remains after that, send us a note and we'll review it with you.",
    },
    {
        "ticket_id": "T-1008",
        "issue_type": "shipment_delay",
        "channel": "email",
        "region": "us",
        "customer_message": "The tracking page hasn't updated in four days and this was a birthday gift.",
        "facts": [
            "Latest carrier scan was four days ago.",
            "Support can open a carrier inquiry after 72 hours without movement.",
            "Replacement eligibility depends on carrier outcome.",
        ],
        "target_response": "I'm sorry this shipment delay is affecting a gift order. Since the tracking has not moved for four days, we can open a carrier inquiry from our side. That is the right next step to understand whether the package is delayed or lost in transit. Once we have the carrier's update, we can advise on replacement options if needed. We'll stay with you on it.",
    },
    {
        "ticket_id": "T-1009",
        "issue_type": "billing_error",
        "channel": "email",
        "region": "eu",
        "customer_message": "I upgraded once, but my invoice shows two plan changes and the total looks wrong.",
        "facts": [
            "Invoice reflects one upgrade and one prorated adjustment line.",
            "The customer was not billed for two full upgrades.",
            "Support can send an itemized invoice explanation.",
        ],
        "target_response": "I'm sorry the invoice wording is confusing. I checked the billing details and it reflects one upgrade plus a prorated adjustment line, not two full upgrade charges. We can send you an itemized explanation if that would help clarify the total. If you'd like, I'll outline the invoice lines in the next message.",
    },
    {
        "ticket_id": "T-1010",
        "issue_type": "password_reset",
        "channel": "email",
        "region": "apac",
        "customer_message": "The reset link says expired as soon as I click it.",
        "facts": [
            "Reset links expire after 30 minutes.",
            "Only the most recent reset link remains valid.",
            "Requesting a fresh link invalidates older ones.",
        ],
        "target_response": "I'm sorry the reset link expired before you could use it. Those links stay active for 30 minutes, and only the most recent one remains valid. Please request a new reset email and use the latest link only. If the new link fails too, reply here and we'll help you troubleshoot it with you.",
    },
    {
        "ticket_id": "T-1011",
        "issue_type": "account_lock",
        "channel": "chat",
        "region": "eu",
        "customer_message": "I keep getting locked out after single sign-on. Is something broken on your side?",
        "facts": [
            "No active platform-wide SSO incident is reported.",
            "Multiple failed SSO attempts can still trigger a temporary lock.",
            "Support can collect timestamp and IdP details for investigation.",
        ],
        "target_response": "I'm sorry you're being locked out repeatedly. We don't currently see a platform-wide SSO incident, but repeated failed sign-in attempts can still trigger a temporary lock. If you can share the time of the latest attempt and your identity provider, we can investigate the pattern with more detail. We'll help you narrow it down.",
    },
    {
        "ticket_id": "T-1012",
        "issue_type": "feature_request",
        "channel": "email",
        "region": "us",
        "customer_message": "Can your API return webhook delivery history? We need it for audits.",
        "facts": [
            "Webhook delivery history is available in the admin UI, not yet via API.",
            "Customers can export audit logs from the UI.",
            "The API request has been raised internally before.",
        ],
        "target_response": "Thanks for raising this use case. Webhook delivery history is currently available in the admin UI rather than through the API. If you need the data now, you can export the audit logs from the UI. I've also added your need to the existing internal request for API support. If useful, I can point you to the audit export steps.",
    },
]

df = pd.DataFrame(records)
print(df[["ticket_id", "issue_type", "channel", "region"]])
print(f"
Rows: {len(df)}")

In [ ]:
# Quick inspection of the raw supervised dataset
sample = df.loc[0, ["customer_message", "facts", "target_response"]].to_dict()
print(json.dumps(sample, indent=2))

## 4. Format Examples for Supervised Fine-Tuning

For style tuning, the target should be the **assistant reply only**. Do not train the model to emit hidden metadata, labels, or internal policy notes unless you truly want that in production outputs.

A good SFT example contains:

- a **system message** with the support style rules
- a **user message** containing the customer message and structured facts
- an **assistant message** containing the ideal reply

### Why Structured Facts Matter

Support style tuning fails when the model cannot tell which text is customer-provided versus internally verified. Keep facts explicitly separated to reduce hallucination and make evaluation easier.

In [ ]:
def build_user_prompt(row):
        facts_block = "
".join(f"- {fact}" for fact in row["facts"])
        return f'''Customer message:
    {row['customer_message']}

    Verified case facts:
    {facts_block}

    Write the customer-facing support response.'''


def to_chat_record(row):
    return {
        "ticket_id": row["ticket_id"],
        "issue_type": row["issue_type"],
        "channel": row["channel"],
        "region": row["region"],
        "messages": [
            {"role": "system", "content": STYLE_SYSTEM_PROMPT},
            {"role": "user", "content": build_user_prompt(row)},
            {"role": "assistant", "content": row["target_response"]},
        ],
    }

chat_records = [to_chat_record(row) for _, row in df.iterrows()]
chat_df = pd.DataFrame(chat_records)
print(chat_df[["ticket_id", "issue_type", "channel", "region"]].head(3))
print("
Example messages:
")
print(json.dumps(chat_records[0]["messages"], indent=2))

In [ ]:
# Save a JSONL version that can be reused outside this notebook
jsonl_path = ARTIFACT_DIR / "support_style_sft.jsonl"
with jsonl_path.open("w", encoding="utf-8") as handle:
    for record in chat_records:
        handle.write(json.dumps(record, ensure_ascii=False) + "
")

print(f"Saved: {jsonl_path}")

## 5. Train / Validation Split Design

A random split is often too optimistic for support data.

### Common Leakage Mistakes

- splitting individual turns from the same customer thread across train and validation
- letting near-duplicate macros appear in both sets
- mixing older and newer policy eras randomly
- reusing rewritten variants of the same reply in both sets

### Better Split Rules

For real support data, prefer one of these:

- **conversation-level split**: keep a whole thread in one side only
- **time-based split**: train on older periods, validate on newer periods
- **customer/account split**: keep one account entirely in train or validation
- **policy-era split**: isolate before/after major support policy changes

This demo uses a stratified split by `issue_type` only because the dataset is tiny and synthetic. In production, stratification is helpful, but leakage prevention matters more.

In [ ]:
train_df, val_df = train_test_split(
    chat_df,
    test_size=0.25,
    random_state=SEED,
    stratify=chat_df["issue_type"],
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print(f"Train rows: {len(train_df)}")
print(f"Validation rows: {len(val_df)}")
print("
Train issue mix:")
print(train_df["issue_type"].value_counts())
print("
Validation issue mix:")
print(val_df["issue_type"].value_counts())

In [ ]:
# Small split diagnostics
summary = pd.DataFrame(
    {
        "train": train_df["issue_type"].value_counts(normalize=True).sort_index(),
        "validation": val_df["issue_type"].value_counts(normalize=True).sort_index(),
    }
).fillna(0)
summary["abs_gap"] = (summary["train"] - summary["validation"]).abs()
summary

## 6. Dataset Quality Checks Before Training

Do not train until the dataset passes basic hygiene checks.

### Minimum Checks

- target responses exist and are not empty
- replies are not absurdly short or long
- no placeholders like `[[agent_name]]` remain
- no internal-only notes leak into assistant targets
- style is reasonably consistent across examples

For support workloads, add a human pass on a sample of records before training. Bad exemplars do not average out; they become model behavior.

In [ ]:
def quality_checks(frame):
    issues = []
    for _, row in frame.iterrows():
        target = row["messages"][-1]["content"]
        if not target.strip():
            issues.append((row["ticket_id"], "empty_target"))
        if len(target.split()) < 20:
            issues.append((row["ticket_id"], "too_short"))
        if "[[" in target or "]]" in target:
            issues.append((row["ticket_id"], "placeholder_leak"))
        if re.search(r"internal note|jira|zendesk macro", target, flags=re.I):
            issues.append((row["ticket_id"], "internal_text_leak"))
    return pd.DataFrame(issues, columns=["ticket_id", "issue"]) if issues else pd.DataFrame(columns=["ticket_id", "issue"])

quality_report = quality_checks(chat_df)
print("Quality issues found:" if len(quality_report) else "No quality issues found")
quality_report

## 7. Prepare Chat Text for the Trainer

Most modern instruction-tuned models expect a chat template. We convert the message list into one `text` field per example.

### Model Choice

This notebook uses a small instruct model plus LoRA:

- base model: `Qwen/Qwen2.5-3B-Instruct`
- adaptation: LoRA
- target: support tone and reply structure

This is usually enough for style tuning. You do not need a massive model just to learn a stable support voice.

In [ ]:
from transformers import AutoTokenizer


tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def render_training_text(messages):
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

train_dataset = train_dataset.map(lambda row: {"text": render_training_text(row["messages"])})
val_dataset = val_dataset.map(lambda row: {"text": render_training_text(row["messages"])})

print(train_dataset[0]["text"][:1200])

In [ ]:
# Inspect sequence lengths before training
train_lengths = [len(tokenizer(example["text"]).input_ids) for example in train_dataset]
val_lengths = [len(tokenizer(example["text"]).input_ids) for example in val_dataset]

length_stats = pd.DataFrame(
    {
        "split": ["train", "validation"],
        "count": [len(train_lengths), len(val_lengths)],
        "mean_tokens": [np.mean(train_lengths), np.mean(val_lengths)],
        "p95_tokens": [np.percentile(train_lengths, 95), np.percentile(val_lengths, 95)],
        "max_tokens": [np.max(train_lengths), np.max(val_lengths)],
    }
)
length_stats

## 8. Fine-Tuning Configuration

This section sets up a parameter-efficient supervised fine-tuning run.

### Practical Defaults

- LoRA instead of full fine-tuning
- small batch size with gradient accumulation
- 1 to 3 epochs for style learning
- validation every few steps
- keep the base model frozen except adapter layers

For style improvement, overtraining is a real risk. The model can become too templated or too eager to apologize. Use short runs and validate often.

In [ ]:
import torch
from peft import LoraConfig
from transformers import AutoModelForCausalLM, TrainingArguments
from trl import SFTTrainer

USE_4BIT = torch.cuda.is_available()
MAX_SEQ_LENGTH = int(max(length_stats["max_tokens"].max() + 64, 512))

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=2,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    weight_decay=0.01,
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=5,
    save_strategy="epoch",
    save_total_limit=2,
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    seed=SEED,
)

print(peft_config)
print(training_args)

In [ ]:
# Build the model and trainer.
# If 4-bit loading fails in your environment, remove quantization_config and retry with full precision.

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else (torch.float16 if torch.cuda.is_available() else torch.float32),
    device_map="auto" if torch.cuda.is_available() else None,
)
model.config.use_cache = False

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
)

print("Trainer ready")
print(f"Max sequence length: {MAX_SEQ_LENGTH}")

In [ ]:
if RUN_TRAINING:
    train_result = trainer.train()
    trainer.save_model(str(OUTPUT_DIR))
    tokenizer.save_pretrained(str(OUTPUT_DIR))
    print(train_result)
    print(f"Saved adapter to: {OUTPUT_DIR}")
else:
    print("Training skipped. Set RUN_TRAINING = True to run LoRA fine-tuning.")

## 9. Offline Evaluation Design

Fine-tuning should be evaluated against a **clear baseline**. For style tuning, use a layered evaluation design.

### Layer 1: Holdout Validation Set

Use unseen tickets and compare:

- base model with your standard prompt
- fine-tuned model with the same factual inputs

### Layer 2: Style Rubric

Score each answer on dimensions like:

- empathy in the opening
- clarity of next step
- groundedness to provided facts
- concision
- policy-safe wording

### Layer 3: Human Review

Support leads or QA reviewers should score blind pairs. This matters because style quality is partly subjective and customer-facing.

### Layer 4: Online Experiment

Once offline results look good, run an A/B or shadow test in production metrics such as CSAT, reopen rate, escalation rate, or average handling time.

### Important Rule

Do **not** judge the fine-tuned model on examples that were paraphrased from the training set. That inflates results and hides overfitting.

In [ ]:
STYLE_RUBRIC = pd.DataFrame(
    [
        {"criterion": "Acknowledges issue early", "weight": 0.20, "success_signal": "Opening sentence recognizes customer problem"},
        {"criterion": "Clear next step", "weight": 0.25, "success_signal": "Customer knows what to do or what happens next"},
        {"criterion": "Grounded in facts", "weight": 0.25, "success_signal": "No invented policy or unsupported claims"},
        {"criterion": "Concise and readable", "weight": 0.15, "success_signal": "Useful without being verbose"},
        {"criterion": "Calm brand tone", "weight": 0.15, "success_signal": "Warm, non-defensive, professional"},
    ]
)
STYLE_RUBRIC

In [ ]:
def heuristic_style_score(text, facts):
    text_lower = text.lower()
    score = 0.0

    acknowledges = any(phrase in text_lower for phrase in ["i'm sorry", "i understand", "sorry", "thanks for", "i know that's inconvenient"])
    next_step = any(phrase in text_lower for phrase in ["if", "please", "reply", "we can", "the next step"])
    concise = 35 <= len(text.split()) <= 110
    grounded = sum(1 for fact in facts if any(token.lower() in text_lower for token in re.findall(r"[A-Za-z0-9@%-]+", fact)[:4])) >= 1
    blame_free = not any(term in text_lower for term in ["you should have", "you failed to", "your mistake"])

    score += 0.20 if acknowledges else 0.0
    score += 0.25 if next_step else 0.0
    score += 0.25 if grounded else 0.0
    score += 0.15 if concise else 0.0
    score += 0.15 if blame_free else 0.0
    return round(score, 3)

validation_eval = val_df.copy()
validation_eval["reference_score"] = [
    heuristic_style_score(row["messages"][-1]["content"], df.set_index("ticket_id").loc[row["ticket_id"], "facts"])
    for _, row in val_df.iterrows()
]
validation_eval[["ticket_id", "issue_type", "reference_score"]]

In [ ]:
from transformers import pipeline
from peft import PeftModel


def build_eval_prompt(messages):
    return messages[:-1]


def load_generation_pipeline(model_path_or_name):
    generation_model = AutoModelForCausalLM.from_pretrained(
        model_path_or_name,
        trust_remote_code=True,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else (torch.float16 if torch.cuda.is_available() else torch.float32),
        device_map="auto" if torch.cuda.is_available() else None,
    )
    return pipeline("text-generation", model=generation_model, tokenizer=tokenizer)


def generate_reply(generator, messages, max_new_tokens=180):
    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    output = generator(prompt_text, max_new_tokens=max_new_tokens, do_sample=False, temperature=None)
    generated = output[0]["generated_text"]
    return generated[len(prompt_text):].strip()

if RUN_OFFLINE_GENERATION_EVAL:
    base_generator = load_generation_pipeline(BASE_MODEL)

    tuned_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, trust_remote_code=True)
    tuned_model = PeftModel.from_pretrained(tuned_model, str(OUTPUT_DIR))
    tuned_generator = pipeline("text-generation", model=tuned_model, tokenizer=tokenizer)

    eval_rows = []
    facts_lookup = df.set_index("ticket_id")["facts"].to_dict()
    for _, row in val_df.iterrows():
        prompt_messages = build_eval_prompt(row["messages"])
        reference = row["messages"][-1]["content"]
        base_reply = generate_reply(base_generator, prompt_messages)
        tuned_reply = generate_reply(tuned_generator, prompt_messages)
        eval_rows.append(
            {
                "ticket_id": row["ticket_id"],
                "issue_type": row["issue_type"],
                "reference": reference,
                "base_reply": base_reply,
                "tuned_reply": tuned_reply,
                "base_style_score": heuristic_style_score(base_reply, facts_lookup[row["ticket_id"]]),
                "tuned_style_score": heuristic_style_score(tuned_reply, facts_lookup[row["ticket_id"]]),
            }
        )

    eval_df = pd.DataFrame(eval_rows)
    eval_df["lift"] = eval_df["tuned_style_score"] - eval_df["base_style_score"]
    display(eval_df[["ticket_id", "issue_type", "base_style_score", "tuned_style_score", "lift"]])
    print(f"Mean style lift: {eval_df['lift'].mean():.3f}")
else:
    print("Offline generation eval skipped. Set RUN_OFFLINE_GENERATION_EVAL = True after training adapters.")

## 10. Human Review and Online Evaluation

Offline heuristics are not enough. A production-ready evaluation plan should include both human review and online metrics.

### Blind Human Review

For each validation ticket, show reviewers:

- response A
- response B
- the verified case facts
- the rubric

Hide which model produced which answer. Ask reviewers to score:

- empathy
- clarity
- correctness relative to facts
- brand fit
- customer usefulness

### Suggested Online Metrics

- CSAT or thumbs-up rate
- reopen rate within 7 days
- escalation rate to human agent
- average handling time
- percentage of responses requiring manual rewrite

### Rollout Pattern

1. shadow mode on live tickets
2. internal QA review
3. small traffic slice
4. broader A/B only if offline and QA results are positive

In [ ]:
human_eval_plan = pd.DataFrame(
    [
        {"stage": "Offline blind review", "sample_size": 100, "owner": "QA lead", "success_threshold": ">= 55% preference over baseline"},
        {"stage": "Shadow mode", "sample_size": 500, "owner": "Support ops", "success_threshold": "No increase in policy errors"},
        {"stage": "Small A/B", "sample_size": "2-5% traffic", "owner": "Support + analytics", "success_threshold": "CSAT up, reopen rate flat or down"},
        {"stage": "Scale-up", "sample_size": "25%+ traffic", "owner": "Support leadership", "success_threshold": "Sustained gain for 2+ weeks"},
    ]
)
human_eval_plan

## 11. When Fine-Tuning Is Worth It

Use a decision scorecard before committing to training and maintenance.

Fine-tuning is usually worth it when:

- style inconsistency is a persistent business problem
- prompt-only fixes are too long or still brittle
- you have enough high-quality labeled examples
- the support tone is stable enough to encode into weights
- you can evaluate with real business metrics

It is usually **not** worth it when:

- your main issue is outdated knowledge
- you cannot define a clear gold style
- policy or product behavior changes constantly
- the support team rewrites every AI response anyway
- the improvement would not change customer or ops outcomes

In [ ]:
decision_scorecard = pd.DataFrame(
    [
        {"question": "Do we have 500+ high-quality exemplar replies?", "why_it_matters": "Weights need consistent supervision", "yes_signal": "Plenty of edited gold responses", "no_signal": "Too little clean data"},
        {"question": "Is the desired style stable for at least a quarter?", "why_it_matters": "Frequent style changes create retraining churn", "yes_signal": "Brand voice and support standards are stable", "no_signal": "Guidance changes every few weeks"},
        {"question": "Did prompt engineering plateau?", "why_it_matters": "Fine-tuning should beat a strong prompt baseline", "yes_signal": "Prompt still drifts or is too verbose", "no_signal": "Prompt already solves the issue"},
        {"question": "Can we run blind human evals and A/B tests?", "why_it_matters": "You need evidence beyond vibes", "yes_signal": "QA + analytics are available", "no_signal": "No evaluation path"},
        {"question": "Would consistency gains affect CSAT, QA load, or handling time?", "why_it_matters": "The gain must matter operationally", "yes_signal": "Clear business upside", "no_signal": "Nice-to-have only"},
    ]
)
decision_scorecard

## 12. Practical Recommendations

### Recommended Starting Path

1. write a short style guide
2. build a strong prompt baseline first
3. collect or clean a gold dataset of ideal replies
4. split by thread, account, or time to avoid leakage
5. run a small LoRA experiment
6. compare against the prompt baseline with blind review
7. only ship if business metrics improve

### Final Rule of Thumb

If the problem is **voice, consistency, or response framing**, fine-tuning can be worth it.

If the problem is **missing knowledge, changing policy, or business logic**, use retrieval, tools, and validators first.

That division keeps the model focused on stable behavior and keeps volatile facts outside the weights.